## **Objectif**

 Assembler les briques des étapes 2 à 4 en **un seul objet** `RagPipeline`, utilisé tel quel à l'étape 6 pour les configurations **B** (Qwen + RAG) et **C** (Qwen + RAG + adaptateur QLoRA). Le système retourne, pour chaque question, une réponse **avec les passages sources** ou une **abstention explicite**.

**Ce que ce notebook valide : l'intégration, pas la performance.** Le jeu de test n'est **pas** utilisé ici (il l'est uniquement à l'étape 6).
Les contrôles sont des tests de cohérence : (1) le retrieval rechargé redonne les scores de l'étape 2 sur le dev ; (2) le prompt reconstruit ici est **identique, caractère pour caractère,** à celui de l'entraînement ; (3) la logique de citation / d'abstention se comporte comme prévu sur des cas du dev.

### Architecture : trois niveaux d'abstention
```
question ─► retriever (FAISS, top-k) ─► [1] garde-fou retrieval : cosinus du meilleur passage < PLANCHER ? ─► ABSTENTION (sans appeler le LLM)
                                    └► prompt RAG (passages numérotés [1]…[k]) ─► LLM (B : adaptateur désactivé | C : adaptateur actif)
                                         ├► [2] le LLM s'abstient (« INFORMATION ABSENTE ») ─► ABSTENTION
                                         └► [3] réponse sans citation valide ─► REJET (traité comme abstention, `enforce_citation=True`)
                                         sinon : réponse + passages cités traduits en fiche / page
```
- **[1] Plancher de similarité** : ne coupe que les questions *clairement* hors sujet. Calibré sur le **dev** (jamais sur le test). Le seuil de Youden de l'étape 2
  (0,61) n'est **pas** utilisé comme garde-fou : sur le dev il rejette déjà 19 % des questions en corpus, un coût trop élevé. Il reste un diagnostic.
- **[2] Abstention décidée par le LLM** : c'est ce que l'adaptateur QLoRA a appris. Seule voie pour les questions « proches du sujet » (le cosinus ne les sépare pas).
- **[3] Citation obligatoire** : une réponse sans ligne `Sources : [n]` valide (numéros présents dans le contexte) n'est pas « sourcée » : elle est rejetée.

### Citation vérifiable
Le modèle cite des **numéros de passage** (`[2]`), jamais des numéros de page qu'il pourrait inventer. Le pipeline traduit `[2]` en fiche / section / pages à partir des
métadonnées du chunk. Les passages cités sont retournés avec leur texte.

### Équité B / C
**Un seul modèle chargé** (4-bit NF4) : B = adaptateur LoRA désactivé, C = adaptateur actif. Mêmes poids de base, même prompt, même retrieval, même décodage glouton.
Seule différence : l'adaptateur. Tout écart entre B et C lui est donc attribuable.

### Limites annoncées d'emblée
- Le LLM tourne en **NF4** ; le modèle de l'étape 4 a été entraîné via Unsloth : l'adaptateur est chargé ici avec `peft` standard (même format), la quantification est la même
  mais pas garantie bit à bit identique.
- Génération **par lots** (padding à gauche) : de très petites différences numériques avec une génération question par question sont possibles.
- Pas de mémoire de conversation, pas de reformulation de la question : une question = une recherche.

## 1. Installation et imports

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy

In [ ]:
# !pip install -q -U transformers peft accelerate bitsandbytes sentence-transformers faiss-cpu pandas numpy

In [ ]:
import os, re, json, math, time, gc, random, hashlib, unicodedata, textwrap, platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

SEED = 42 # Fixation de la graine aléatoire pour la reproductibilité
random.seed(SEED); np.random.seed(SEED)
print("python", platform.python_version())

python 3.13.15


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 2. Configuration

Les chemins pointent vers les sorties des étapes 2, 3 et 4. Sur Colab, copier ces dossiers depuis Google Drive (la session est éphémère).
**À adapter** : `MODEL_ID` doit être le même que celui de l'étape 4 ; `BATCH_SIZE` à baisser en cas de `CUDA out of memory`.

In [ ]:
STEP2_DIR = Path("outputs/step2_index")
STEP3_DIR = Path("outputs/step3_dataset")
STEP4_DIR = Path("outputs/step4_qlora")
ADAPTER_DIR = STEP4_DIR / "adapter"
DEV_SET_PATH = Path("dev_retrieval_b1.jsonl")
OUTPUT_DIR = Path("outputs/step5_pipeline"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MAX_NEW_TOKENS = 300        # identique à l'étape 1 (baseline A)
BATCH_SIZE = 4              # identique à l'étape 1 ; le garder pour B et C
ENFORCE_CITATION = True     # une réponse sans citation valide est rejetée (traitée comme abstention) ; la sortie brute du modèle est conservée

def load_jsonl(path):
    # Charge un fichier JSONL ligne par ligne en ignorant les lignes vides
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

## 3. Chargement des artefacts des étapes précédentes et contrôles de cohérence

On recharge **depuis le disque** : l'index FAISS, les chunks, la configuration retenue (embedder, méthode, `k`), et le prompt RAG de l'étape 3.
Les assertions échouent bruyamment si les étapes ne sont pas cohérentes entre elles (par exemple un index refait sans refaire le dataset).

In [ ]:
index_cfg = json.load(open(STEP2_DIR / "index_config.json", encoding="utf-8"))
rag_prompt = json.load(open(STEP3_DIR / "rag_prompt.json", encoding="utf-8"))
dataset_cfg = json.load(open(STEP3_DIR / "dataset_config.json", encoding="utf-8"))
chunks = load_jsonl(STEP2_DIR / "chunks.jsonl")
chunk_by_id = {c["chunk_id"]: c for c in chunks}
K = int(index_cfg["k_final"])
ABSTAIN_MARKER, ABSTAIN_ANSWER = rag_prompt["abstain_marker"], rag_prompt["abstain_answer"]

# Vérification de la cohérence stricte entre l'index (étape 2) et le jeu de données (étape 3)
assert rag_prompt["k_context"] == K, f"k de l'étape 3 ({rag_prompt['k_context']}) ≠ k_final de l'étape 2 ({K}) : refaire l'étape 3 puis l'étape 4"
assert dataset_cfg["step2_config"]["embedder"] == index_cfg["embedder"] and dataset_cfg["step2_config"]["chunking"] == index_cfg["chunking"], \
    "le dataset de l'étape 3 a été construit avec un autre index que celui chargé ici"
assert len(chunks) == index_cfg["n_chunks"], "chunks.jsonl ne correspond pas à index_config.json"
print(f"{len(chunks)} chunks | embedder={index_cfg['embedder']} méthode={index_cfg['method']} k={K} | abstention : « {ABSTAIN_MARKER} »")

239 chunks | embedder=bge-m3 méthode=dense k=5 | abstention : « INFORMATION ABSENTE »


## 4. Retriever

Reprise du `Retriever` de l'étape 2 (mêmes normalisation, mêmes préfixes d'embedding, même fusion si la méthode retenue est hybride).
Il renvoie, pour chaque passage, ses métadonnées de citation ; le cosinus du meilleur passage (`top1_cosine`) sert au garde-fou [1].

In [ ]:
def norm(s):
    # Passage en minuscules, harmonisation des apostrophes et suppression des accents
    s = unicodedata.normalize("NFKD", s.lower().replace("’", "'"))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s  ]+", " ", s).strip()

STOPWORDS = set("le la les l un une des de du d et ou a au aux en dans par pour sur sous avec sans que qui quoi est sont etre ce cet cette ces se sa son ses "
                "leur leurs ne pas plus il ils elle elles on y c s n qu j m t si comme mais donc car faut faire peut doit quel quelle quels quelles".split())
def tokenize(text):
    toks = re.findall(r"[a-z0-9]+", norm(text))
    # Suppression des mots vides et retrait naïf du 's' final pour les mots de plus de 3 lettres
    return [t[:-1] if len(t) > 3 and t.endswith("s") else t for t in toks if t not in STOPWORDS]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = [Counter(d) for d in docs_tokens]
        self.len = np.array([len(d) for d in docs_tokens], dtype=float); self.avg = self.len.mean()
        df = Counter(t for d in self.docs for t in d); n = len(docs_tokens)
        self.idf = {t: math.log(1 + (n - f + 0.5) / (f + 0.5)) for t, f in df.items()}
    def scores(self, q_tokens):
        s = np.zeros(len(self.docs))
        for t in set(q_tokens):
            if t in self.idf:
                tf = np.array([d.get(t, 0) for d in self.docs], dtype=float)
                s += self.idf[t] * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len / self.avg))
        return s

def rrf_fuse(rank_lists, k=60, topn=10):
    # Reciprocal Rank Fusion (RRF) : combine plusieurs listes de rangs sans normaliser les scores bruts
    sc = defaultdict(float)
    for ranks in rank_lists:
        for pos, j in enumerate(ranks):
            sc[int(j)] += 1.0 / (k + pos + 1)
    return [j for j, _ in sorted(sc.items(), key=lambda x: -x[1])[:topn]]

def cite(c):
    """Chaîne de citation d'un chunk. IDENTIQUE à l'étape 3 (c'est elle qui apparaît dans les prompts d'entraînement)."""
    p = [x for x in c["pages"] if x >= 1]            # la page de présentation du PDF n'a pas de numéro imprimé
    pages = "page non numérotée" if not p else (f"p. {p[0]}" if len(p) == 1 else f"p. {p[0]}-{p[-1]}")
    where = "Présentation du guide" if c["fiche"] == 0 else f"Fiche {c['fiche']} — {c['fiche_title']}"
    return f"{where} — {c['section_label']}, {pages}"

EMBEDDERS = {
    "e5-base":  {"hf_id": "intfloat/multilingual-e5-base",  "q": "query: ", "p": "passage: "},
    "e5-large": {"hf_id": "intfloat/multilingual-e5-large", "q": "query: ", "p": "passage: "},
    "bge-m3":   {"hf_id": "BAAI/bge-m3",                    "q": "",        "p": ""},
}

class STEmbedder:
    """Enveloppe sentence-transformers (même code qu'à l'étape 2). Vecteurs normalisés : produit scalaire = cosinus."""
    def __init__(self, name):
        import torch
        from sentence_transformers import SentenceTransformer
        self.name, self.cfg = name, EMBEDDERS[name]
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(self.cfg["hf_id"], device=self.device)
        self.model.max_seq_length = 512
        # Passage en demi-précision (FP16) sur GPU pour accélérer l'encodage
        if self.device == "cuda":
            self.model.half()
    def _enc(self, texts, prefix):
        return self.model.encode([prefix + t for t in texts], batch_size=32, normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=False).astype("float32")
    def encode_queries(self, texts): return self._enc(texts, self.cfg["q"])

def make_embedder(name):
    return STEmbedder(name)

class Retriever:
    def __init__(self, out_dir, embedder=None):
        import faiss
        out_dir = Path(out_dir)
        self.cfg = json.load(open(out_dir / "index_config.json", encoding="utf-8"))
        self.chunks = load_jsonl(out_dir / "chunks.jsonl")
        self.index = faiss.read_index(str(out_dir / "faiss.index"))
        assert self.index.ntotal == len(self.chunks), "index FAISS et chunks désynchronisés"
        self.embedder = embedder or make_embedder(self.cfg["embedder"])
        self.bm25 = BM25([tokenize(c["embed_text"]) for c in self.chunks]) if self.cfg["method"] == "hybrid" else None
        self.k = self.cfg["k_final"]
    def search_batch(self, questions, k=None):
        k = k or self.k
        n = min(50, len(self.chunks))
        Q = self.embedder.encode_queries(list(questions))
        scores, ids = self.index.search(Q, n)
        out = []
        for qi, (q, s_row, i_row) in enumerate(zip(questions, scores, ids)):
            if self.bm25 is not None:
                bm_rank = np.argsort(-self.bm25.scores(tokenize(q)))[:n]
                order = rrf_fuse([i_row, bm_rank], k=self.cfg["rrf_k"], topn=k)
            else:
                order = [int(j) for j in i_row[:k]]
            cos = dict(zip(i_row.tolist(), s_row.tolist()))
            res = []
            for rank, j in enumerate(order, 1):
                c = self.chunks[j]
                # Si le chunk provient uniquement du classement BM25 (absent du top-n FAISS), on recalcule son cosinus exact
                cj = cos[int(j)] if int(j) in cos else float(Q[qi] @ self.index.reconstruct(int(j)))
                res.append({"rank": rank, "chunk": c, "cosine": cj})
            out.append({"question": q, "top1_cosine": float(s_row[0]), "results": res})
        return out

## 5. Prompt RAG (source de vérité : `rag_prompt.json` de l'étape 3)

On **recharge** le prompt au lieu de le recopier. Contrôle fort : pour des exemples du `val.jsonl` de l'étape 3, on reconstruit les messages à partir des identifiants de
chunks (`meta.context_chunks`) et on exige l'égalité exacte avec les messages enregistrés. Si cette assertion passe, le modèle reçoit en production
exactement le format sur lequel il a été entraîné.

In [ ]:
def build_messages(question, passages):
    """passages : liste de chunks (dict), dans l'ordre d'affichage [1], [2]…"""
    ctx = "\n\n".join(rag_prompt["passage_template"].format(n=i + 1, citation=cite(c), text=c["text"]) for i, c in enumerate(passages))
    return [{"role": "system", "content": rag_prompt["system"]},
            {"role": "user", "content": rag_prompt["user_template"].format(passages=ctx, question=question)}]

val_records = load_jsonl(STEP3_DIR / "val.jsonl")
n_checked = 0
for rec in val_records:
    # Extraction de la question brute depuis le message utilisateur formaté de l'étape 3
    q = rec["messages"][1]["content"].split("Question :")[-1].strip()
    rebuilt = build_messages(q, [chunk_by_id[cid] for cid in rec["meta"]["context_chunks"]])
    assert rebuilt == rec["messages"][:2], f"prompt reconstruit ≠ prompt d'entraînement pour {rec['meta']['id']}"
    n_checked += 1
print(f"OK : {n_checked} prompts du val reconstruits à l'identique de l'étape 3.")

OK : 28 prompts du val reconstruits à l'identique de l'étape 3.


## 6. Générateur : Qwen 4-bit + adaptateur LoRA (un seul modèle, B = adaptateur désactivé, C = adaptateur actif)

- Quantification **NF4 + double quantification**, comme aux étapes 1 et 4.
- **GPU requis** (CUDA). Sur T4 : `float16` (pas de bf16 natif). Sur GPU Ampere ou plus récent : `bfloat16`.
- Décodage **glouton**, mêmes paramètres que la baseline A.
- `torch` / `transformers` ne sont importés qu'ici : le reste du pipeline est testable sans GPU (cf. section 10).

In [ ]:
class QwenGenerator:
    def __init__(self, model_id=MODEL_ID, adapter_dir=ADAPTER_DIR, max_new_tokens=MAX_NEW_TOKENS, batch_size=BATCH_SIZE):
        import torch, transformers
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from peft import PeftModel
        if not torch.cuda.is_available():
            raise RuntimeError("Aucun GPU CUDA : bitsandbytes (4-bit) ne fonctionne que sur NVIDIA. Utiliser Colab/Kaggle, ou une autre chaîne d'inférence (llama.cpp / MLX).")
        self.torch, self.max_new_tokens, self.batch_size = torch, max_new_tokens, batch_size
        # Ampere (RTX 30xx, A100) et plus récents supportent bfloat16 ; T4/V100 utilisent float16
        compute = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute)
        self.tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")      # left padding : requis pour la génération par lots
        if self.tok.pad_token is None: self.tok.pad_token = self.tok.eos_token
        try:
            base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, dtype=compute, device_map={"": 0})
        except TypeError:                                                           # anciennes versions de transformers
            base = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, torch_dtype=compute, device_map={"": 0})
        self.model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)
        self.model.eval()
        # Empreinte cryptographique des poids LoRA pour garantir la traçabilité à l'étape 6
        self.adapter_sha256 = hashlib.sha256((Path(adapter_dir) / "adapter_model.safetensors").read_bytes()).hexdigest()
        print(f"modèle chargé | dtype de calcul {compute} | VRAM utilisée {torch.cuda.memory_allocated() / 1e9:.1f} Go | adaptateur sha256 {self.adapter_sha256[:12]}…")

    def __call__(self, list_of_messages, adapter):
        """adapter=True -> configuration C ; adapter=False -> configuration B (LoRA désactivé, poids de base seuls)."""
        outs = []
        for s in range(0, len(list_of_messages), self.batch_size):
            batch = list_of_messages[s:s + self.batch_size]
            prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch]
            enc = self.tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
            # Décodage glouton déterministe (do_sample=False), neutralisation explicite des paramètres d'échantillonnage
            kw = dict(max_new_tokens=self.max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None, pad_token_id=self.tok.pad_token_id)
            with self.torch.inference_mode():
                if adapter:
                    out = self.model.generate(**enc, **kw)
                else:
                    # Désactivation temporaire des poids LoRA pour évaluer le modèle de base seul (configuration B)
                    with self.model.disable_adapter():
                        out = self.model.generate(**enc, **kw)
            outs += [self.tok.decode(g, skip_special_tokens=True).strip() for g in out[:, enc["input_ids"].shape[1]:]]
        return outs

## 7. Citation et abstention : règles de décision

**Lecture de la sortie du modèle** (`parse_output`) :\n
- *abstention* : la sortie **commence** par `INFORMATION ABSENTE` (insensible à la casse ; même si le modèle ajoute ensuite une ligne `Sources`, comme le fait parfois le modèle de base, l'abstention prime) ;
- sinon, on vérifie que la **dernière ligne** de la réponse respecte le format `Sources : [n], [m]` ; tous les numéros cités doivent exister dans le contexte (`1 ≤ n ≤ k`) ;
- une réponse sans citation valide est **rejetée** si `enforce_citation=True` (statut `rejected_citation`) ; la sortie brute est conservée dans `raw_answer` pour que l'étape 6 puisse aussi mesurer le modèle *sans* cette règle.

**Plancher de similarité (garde-fou [1])** calibré sur le **dev** : milieu de l'intervalle entre le meilleur score d'une question *hors sujet* et le plus faible score d'une question *en corpus*. C'est une décision conservatrice : elle ne supprime aucune question en corpus du dev, et ne coupe que les questions manifestement étrangères au guide.
**Limite :** 3 questions hors sujet seulement dans le dev ; le plancher est une estimation grossière, à relire avec les résultats de l'étape 6.

In [ ]:
# Capture une ligne finale de la forme "Sources : [1], [2]" en fin de réponse
SOURCES_RE = re.compile(r"(?:^|\n)\s*Sources\s*:\s*((?:\[\d+\](?:\s*,\s*)?)+)\s*$")

def parse_output(raw, k):
    """-> dict(abstained, cited (liste d'entiers valides), valid_citation, body)."""
    text = raw.strip()
    if text.upper().startswith(ABSTAIN_MARKER):
        return dict(abstained=True, cited=[], valid_citation=False, body=text)
    m = SOURCES_RE.search(text)
    cited = sorted({int(x) for x in re.findall(r"\d+", m.group(1))}) if m else []
    valid = bool(m) and bool(cited) and all(1 <= c <= k for c in cited)
    body = text[:m.start()].strip() if m else text
    return dict(abstained=False, cited=cited if valid else [], valid_citation=valid, body=body)

def calibrate_floor(retriever, dev_rows):
    """Plancher de cosinus calibré sur le dev (voir texte ci-dessus). Retourne (plancher, détails)."""
    ins = [r for r in dev_rows if r["category"] == "in_corpus"]
    offs = [r for r in dev_rows if r["category"] == "out_corpus_offtopic"]
    top_in = np.array([x["top1_cosine"] for x in retriever.search_batch([r["question"] for r in ins], k=1)])
    top_off = np.array([x["top1_cosine"] for x in retriever.search_batch([r["question"] for r in offs], k=1)])
    hi_off, lo_in = float(top_off.max()), float(top_in.min())
    if hi_off >= lo_in:                       # pas d'intervalle de séparation : on désactive le garde-fou plutôt que de perdre des questions en corpus
        return None, dict(max_offtopic=hi_off, min_in_corpus=lo_in, note="recouvrement : garde-fou désactivé")
    return (hi_off + lo_in) / 2, dict(max_offtopic=hi_off, min_in_corpus=lo_in, n_in=len(ins), n_offtopic=len(offs))

## 8. La classe `RagPipeline`

`answer(questions, config="B"|"C")` retourne une liste de dictionnaires :

| Champ | Contenu |
|---|---|
| `status` | `answered` · `abstained_retrieval` (plancher) · `abstained_model` (le LLM s'abstient) · `rejected_citation` (réponse sans citation valide) |
| `answer` | texte final : réponse **sans** la ligne Sources, ou la chaîne d'abstention |
| `raw_answer` | sortie brute du modèle (`None` si le LLM n'a pas été appelé) |
| `sources` | passages **cités** : numéro, `chunk_id`, fiche, titre, section, pages, citation, texte |
| `retrieved` | les k passages fournis au modèle (rang, numéro, `chunk_id`, cosinus, fiche, pages) |
| `top1_cosine`, `latency_s` | diagnostics |

In [ ]:
class RagPipeline:
    def __init__(self, retriever, generator, floor=None, enforce_citation=ENFORCE_CITATION, k=K):
        self.retriever, self.generator, self.floor, self.enforce, self.k = retriever, generator, floor, enforce_citation, k

    @staticmethod
    def _src(n, c):
        return dict(n=n, chunk_id=c["chunk_id"], fiche=c["fiche"], fiche_title=c.get("fiche_title"), section=c.get("section_label"),
                    pages=c["pages"], citation=cite(c), text=c["text"])

    def answer(self, questions, config="C"):
        assert config in ("B", "C"), "config doit valoir 'B' (RAG, adaptateur désactivé) ou 'C' (RAG + adaptateur)"
        t0 = time.time()
        hits = self.retriever.search_batch(questions, k=self.k)
        results = [None] * len(questions)
        todo = []                                                              # indices à envoyer au LLM
        for i, h in enumerate(hits):
            passages = [r["chunk"] for r in h["results"]]
            base = dict(question=questions[i], config=config, top1_cosine=h["top1_cosine"], raw_answer=None, sources=[],
                        retrieved=[dict(rank=r["rank"], n=r["rank"], chunk_id=r["chunk"]["chunk_id"], cosine=r["cosine"], fiche=r["chunk"]["fiche"],
                                        pages=r["chunk"]["pages"], citation=cite(r["chunk"])) for r in h["results"]])
            if self.floor is not None and h["top1_cosine"] < self.floor:       # [1] garde-fou retrieval : pas d'appel au LLM
                results[i] = dict(base, status="abstained_retrieval", answer=ABSTAIN_ANSWER)
            else:
                results[i] = base; todo.append(i)
        # Seules les questions ayant passé le garde-fou retrieval [1] sont envoyées au LLM        
        if todo:
            msgs = [build_messages(questions[i], [r["chunk"] for r in hits[i]["results"]]) for i in todo]
            raws = self.generator(msgs, adapter=(config == "C"))
            for i, raw in zip(todo, raws):
                p = parse_output(raw, self.k)
                passages = [r["chunk"] for r in hits[i]["results"]]
                res = dict(results[i], raw_answer=raw)
                if p["abstained"]:                                              # [2] abstention décidée par le LLM
                    res.update(status="abstained_model", answer=ABSTAIN_ANSWER)
                elif not p["valid_citation"] and self.enforce:                  # [3] pas de citation valide : réponse non sourcée
                    res.update(status="rejected_citation", answer=ABSTAIN_ANSWER)
                else:
                    # Conversion des numéros de passage (base 1 dans le prompt) en indices de liste (base 0)
                    res.update(status="answered", answer=p["body"], sources=[self._src(n, passages[n - 1]) for n in p["cited"]])
                results[i] = res
        dt = (time.time() - t0) / max(len(questions), 1)
        for r in results: r["latency_s"] = dt                                   # moyenne par question du lot (le lot est traité ensemble)
        return results

def show(res, width=420):
    print(f"[{res['config']}] {res['question']}\n  statut={res['status']} | cosinus top-1={res['top1_cosine']:.3f}")
    print("  réponse :", textwrap.shorten(res["answer"].replace("\n", " "), width))
    for s in res["sources"]:
        print(f"  source [{s['n']}] {s['citation']}  ({s['chunk_id']})")
    if res["raw_answer"] and res["status"] in ("abstained_model", "rejected_citation"):
        print("  sortie brute :", textwrap.shorten(res["raw_answer"].replace("\n", " "), 200))

## 9. Construction du pipeline et calibration du plancher (dev)

In [ ]:
pip install --upgrade numpy scipy sentence-transformers

In [ ]:
# Chargement du dev, calibration du seuil de rejet retrieval et instanciation du pipeline complet
dev_rows = load_jsonl(DEV_SET_PATH)
retriever = Retriever(STEP2_DIR)
FLOOR, floor_info = calibrate_floor(retriever, dev_rows)
print("plancher de similarité :", None if FLOOR is None else round(FLOOR, 3), "|", floor_info)
generator = QwenGenerator()
pipe = RagPipeline(retriever, generator, floor=FLOOR)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (19 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

modèle chargé | dtype de calcul torch.float16 | VRAM utilisée 6.9 Go | adaptateur sha256 7fce6aa4246a…


## 10. Contrôles d'intégration (sur le dev, jamais sur le test)

### 10.1 Le retrieval rechargé redonne les scores de l'étape 2

On recalcule `rep@k` (tous les faits-clés dans les k passages) sur le dev et on le compare à la valeur enregistrée dans `index_config.json`.
Un écart important signalerait un index, un embedder ou un prétraitement différent de l'étape 2.

In [ ]:
def rep_at_k(retriever, rows, k):
    ok = []
    for r, h in zip(rows, retriever.search_batch([r["question"] for r in rows], k=k)):
        blob = norm(" ".join(x["chunk"]["text"] for x in h["results"]))
        # Vérifie que chaque fait-clé attendu (via au moins une de ses formulations regex) est présent dans le top-k
        ok.append(all(any(re.search(norm(a), blob) for a in alts) for alts in r["key_facts"]))
    return float(np.mean(ok))

dev_in = [r for r in dev_rows if r["category"] == "in_corpus"]
rep_now = rep_at_k(retriever, dev_in, K)
print(f"rep@{K} dev (recalculé) = {rep_now:.3f} | enregistré à l'étape 2 = {index_cfg['dev_rep@k_final']:.3f}")
assert abs(rep_now - index_cfg["dev_rep@k_final"]) <= 0.03, "le retrieval rechargé diverge de l'étape 2 : vérifier index, embedder, version du dev"

rep@5 dev (recalculé) = 0.984 | enregistré à l'étape 2 = 0.984


### 10.2 Cas de démonstration B vs C (dev) : questions en corpus, multi-faits, hors corpus proches et hors sujet

À lire à la main : c'est le premier regard sur le comportement de bout en bout. **Ce n'est pas une évaluation** (6 à 8 cas).

In [ ]:
# Sélection d'un échantillon stratifié : 3 questions simples en corpus, 2 multi-faits (préfixe 'm'), 2 proches hors-corpus et 1 hors-sujet
pick = [r for r in dev_rows if r["category"] == "in_corpus" and not r["id"].startswith("m")][:3] \
     + [r for r in dev_rows if r["id"].startswith("m")][:2] \
     + [r for r in dev_rows if r["category"] == "out_corpus_near"][:2] \
     + [r for r in dev_rows if r["category"] == "out_corpus_offtopic"][:1]
qs = [r["question"] for r in pick]
res_B, res_C = pipe.answer(qs, "B"), pipe.answer(qs, "C")
for r, b, c in zip(pick, res_B, res_C):
    print(f"\n===== {r['id']} [{r['category']}] fiches de référence {r.get('gold_fiches')}")
    show(b); show(c)


===== d01 [in_corpus] fiches de référence [1]
[B] Quelles fonctions le guide recommande-t-il de nommer pour suivre au quotidien la sécurité et la protection des données ?
  statut=answered | cosinus top-1=0.700
  réponse : Le guide recommande de nommer un délégué à la protection des données (DPD) et un responsable de la sécurité des systèmes d'information (RSSI) pour suivre au quotidien la sécurité et la protection des données.
  source [1] Présentation du guide — Introduction, page non numérotée  (c0000)
  source [2] Présentation du guide — Introduction, p. 2  (c0003)
[C] Quelles fonctions le guide recommande-t-il de nommer pour suivre au quotidien la sécurité et la protection des données ?
  statut=abstained_model | cosinus top-1=0.700
  réponse : INFORMATION ABSENTE : les passages fournis ne contiennent pas cette information.
  sortie brute : INFORMATION ABSENTE : les passages fournis ne contiennent pas cette information.

===== d02 [in_corpus] fiches de référence [1, 14]
[B] Quell

### 10.3 Statistiques de statut sur tout le dev (ce que fait chaque configuration, sans juger l'exactitude)

Répartition des statuts par catégorie. Attendu : en corpus → surtout `answered` ; hors corpus → surtout abstentions. Ce tableau ne mesure **pas** si les réponses sont justes
(c'est l'étape 6, sur le test). Il détecte les comportements aberrants : trop d'abstentions en corpus, ou des réponses « sourcées » hors corpus.

In [ ]:
all_q = [r["question"] for r in dev_rows]
rows_stat = []
for cfg_name in ("B", "C"):
    out = pipe.answer(all_q, cfg_name)
    for r, o in zip(dev_rows, out):
        rows_stat.append(dict(config=cfg_name, category=r["category"], status=o["status"], valid_cit=bool(o["sources"]), latency=o["latency_s"]))
stat = pd.DataFrame(rows_stat)
# Tableau croisé des statuts par configuration et catégorie de question
display(stat.groupby(["config", "category", "status"]).size().unstack(fill_value=0))
print("latence moyenne par question (s) :", stat.groupby("config").latency.mean().round(2).to_dict())

status                      abstained_model  abstained_retrieval  answered  \
config category                                                              
B      in_corpus                          0                    0        37   
       out_corpus_near                    9                    1         0   
       out_corpus_offtopic                0                    3         0   
C      in_corpus                          1                    0        60   
       out_corpus_near                    8                    1         1   
       out_corpus_offtopic                0                    3         0   

status                      rejected_citation  
config category                                
B      in_corpus                           25  
       out_corpus_near                      0  
       out_corpus_offtopic                  0  
C      in_corpus                            1  
       out_corpus_near                      0  
       out_corpus_offtopic                  0

latence moyenne par question (s) : {'B': 4.43, 'C': 3.67}


## 11. Sauvegarde de la configuration du pipeline

On enregistre tout ce qui doit rester **identique** à l'étape 6 (et qui permet de reproduire ou d'expliquer un écart) : index, prompt, adaptateur, paramètres de génération, plancher.

In [ ]:
# Sérialisation des hyperparamètres et empreintes SHA-256 (prompt + adaptateur) pour verrouiller la configuration de l'étape 6

pipeline_cfg = {
    "step": "5_pipeline", "date": time.strftime("%Y-%m-%d %H:%M:%S"), "model": MODEL_ID, "quantization": "nf4 + double quant",
    "retrieval": {"embedder": index_cfg["embedder"], "method": index_cfg["method"], "chunking": index_cfg["chunking"], "k": K, "n_chunks": len(chunks)},
    "prompt_sha256": hashlib.sha256(json.dumps(rag_prompt, sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
    "adapter_sha256": generator.adapter_sha256, "decoding": {"greedy": True, "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BATCH_SIZE},
    "similarity_floor": FLOOR, "floor_calibration": floor_info, "enforce_citation": ENFORCE_CITATION, "seed": SEED,
    "dev_rep@k_recomputed": rep_now,
}
json.dump(pipeline_cfg, open(OUTPUT_DIR / "pipeline_config.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(pipeline_cfg, indent=2, ensure_ascii=False)[:1500])

{
  "step": "5_pipeline",
  "date": "2026-10-23 09:50:53",
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "quantization": "nf4 + double quant",
  "retrieval": {
    "embedder": "bge-m3",
    "method": "dense",
    "chunking": {
      "max_chars": 900,
      "overlap_units": 1
    },
    "k": 5,
    "n_chunks": 239
  },
  "prompt_sha256": "ecb12aa711a8cb472cdeb1e085191a972b208633a7ce82a2b3124b696971e6fe",
  "adapter_sha256": "7fce6aa4246aef1168f7ee708f2003bf54ff11cb5bded45fc6e853c6432e1164",
  "decoding": {
    "greedy": true,
    "max_new_tokens": 300,
    "batch_size": 4
  },
  "similarity_floor": 0.4471016228199005,
  "floor_calibration": {
    "max_offtopic": 0.3486095070838928,
    "min_in_corpus": 0.5455937385559082,
    "n_in": 62,
    "n_offtopic": 3
  },
  "enforce_citation": true,
  "seed": 42,
  "dev_rep@k_recomputed": 0.9838709677419355
}


## 12. Limites méthodologiques

1. **Le plancher de similarité repose sur 3 questions hors sujet** (dev) : estimation grossière. Il ne traite pas les questions proches du sujet, laissées au LLM.
2. **Citation = numéro de passage.** Le pipeline garantit que les passages cités ont été fournis au modèle ; il ne garantit **pas** que la réponse est *fidèle* à ces passages
   (un modèle peut citer le bon passage et paraphraser de travers). C'est mesuré à l'étape 6 (juge + règles).
3. **`enforce_citation`** protège contre les réponses non sourcées, mais peut transformer une réponse correcte sans ligne `Sources` en abstention. Les deux sorties (brute et filtrée) sont conservées.
4. **Un seul tour, une seule recherche.** Pas de reformulation, pas de recherche itérative ; une question composée de plusieurs sous-questions est limitée par le top-k.
5. **k = 5 est un compromis** (complétude vs bruit) choisi à l'étape 2 sur le dev ; les questions à réponses réparties sur plus de 5 passages seront incomplètes.
6. **Adaptateur Unsloth chargé avec `peft` standard** et génération par lots : de petites différences numériques avec l'évaluation de l'étape 4 sont possibles.
7. **Latence mesurée en lot** sur un GPU de type T4 : une indication, pas une mesure de production.
8. **Aucune évaluation sur le test dans ce notebook.** Les cas de la section 10 sont du dev.

**Prochaine étape :** ÉTAPE 6 — évaluation finale A / B / C sur le même jeu de test, avec les mêmes métriques que l'étape 1, plus les métriques de retrieval et d'abstention, puis analyse d'erreurs.